# RSNA Knee Abnormality Detection — EDA v2

Mục tiêu: **hiểu dữ liệu trước khi xây model**. Notebook này tập trung vào label availability, report leakage, series/plane structure, scanner/domain shift, DICOM geometry, slice ordering và decode QC.

**Không train model trong notebook này.** Mặc định `FAST_MODE=True` để chỉ đọc CSV + sample header DICOM, không scan toàn bộ ~820k ảnh.

### Cách dùng
1. Tạo Kaggle Notebook và Add Input của competition **RSNA Knee Abnormality Detection**.
2. Import notebook này và `Run All`.
3. Cuối notebook sẽ sinh `RSNA_Knee_EDA_Artifacts.zip` và `RESULTS_TO_SEND_BACK.txt` trong `/kaggle/working`.
4. Gửi lại ZIP hoặc TXT cho ChatGPT để mình phân tích và khóa baseline.

In [ ]:
from pathlib import Path
from collections import Counter
import os, re, json, hashlib, random, unicodedata, warnings, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

warnings.filterwarnings("ignore")
SEED=42
random.seed(SEED); np.random.seed(SEED)
FAST_MODE=True
MAX_DICOM_SERIES_HEADERS = 6000 if FAST_MODE else None
MAX_SERIES_FILE_COUNTS = 3000 if FAST_MODE else None
MAX_SERIES_SLICE_ORDER_AUDIT = 150 if FAST_MODE else 500
MAX_PIXEL_DECODE = 50 if FAST_MODE else 200

LABELS=["ACL","MCL","Medial Meniscus","Lateral Meniscus","Medial OA","Lateral OA","PF OA","Effusion","Synovitis","Baker's","Contusion","Fracture"]
print("FAST_MODE:",FAST_MODE)

In [ ]:
# ============================================================
# 1. Auto-detect Kaggle competition dataset — FIXED
# ============================================================
from pathlib import Path
import pandas as pd

def find_competition_root():
    kaggle_input = Path("/kaggle/input")

    print("Searching under:", kaggle_input)

    # Recursive search instead of only one folder level
    candidates = list(kaggle_input.rglob("train.csv"))

    print(f"Found {len(candidates)} train.csv candidate(s):")
    for p in candidates:
        print(" -", p)

    valid_roots = []

    for train_csv in candidates:
        root = train_csv.parent

        # RSNA Knee dataset should have these files together
        if not (root / "train_series.csv").exists():
            continue

        try:
            cols = pd.read_csv(train_csv, nrows=2).columns.tolist()

            print("\nChecking:", train_csv)
            print("Columns:", cols)

            # Don't make Report mandatory because schema/version
            # differences should not break path detection.
            required = {"StudyInstanceUID", "ACL", "MCL"}

            if required.issubset(set(cols)):
                valid_roots.append(root)

        except Exception as e:
            print("Could not inspect:", train_csv, e)

    if len(valid_roots) == 0:
        raise FileNotFoundError(
            "Found train.csv files, but none matched the RSNA Knee schema.\n"
            "Check the printed paths/columns above."
        )

    # Prefer directory name containing rsna/knee
    preferred = [
        r for r in valid_roots
        if "rsna" in str(r).lower() or "knee" in str(r).lower()
    ]

    root = preferred[0] if preferred else valid_roots[0]

    print("\n✅ Competition root detected:")
    print(root)

    return root


ROOT = find_competition_root()

TRAIN_CSV = ROOT / "train.csv"
TRAIN_SERIES_CSV = ROOT / "train_series.csv"
TRAIN_DICOM_ROOT = ROOT / "train_series"

TEST_CSV = ROOT / "test.csv"
TEST_SERIES_CSV = ROOT / "test_series.csv"
TEST_DICOM_ROOT = ROOT / "test_series"

print("\nFiles:")
print("train.csv        :", TRAIN_CSV.exists())
print("train_series.csv :", TRAIN_SERIES_CSV.exists())
print("train_series/    :", TRAIN_DICOM_ROOT.exists())
print("test.csv         :", TEST_CSV.exists())
print("test_series.csv  :", TEST_SERIES_CSV.exists())
print("test_series/     :", TEST_DICOM_ROOT.exists())

In [ ]:
train=pd.read_csv(TRAIN_CSV)
series=pd.read_csv(TRAIN_SERIES_CSV)
test=pd.read_csv(TEST_CSV) if TEST_CSV.exists() else None
test_series=pd.read_csv(TEST_SERIES_CSV) if TEST_SERIES_CSV.exists() else None
print('train:',train.shape,'train_series:',series.shape)
print('train cols:',train.columns.tolist())
print('series cols:',series.columns.tolist())
display(train.head(3)); display(series.head(3))

In [ ]:
# Label availability + gold prevalence
assert all(c in train.columns for c in LABELS)
train['n_available_labels']=train[LABELS].notna().sum(axis=1)
gold_mask=train['n_available_labels'].eq(len(LABELS))
partial_mask=train['n_available_labels'].between(1,len(LABELS)-1)
unlabeled_mask=train['n_available_labels'].eq(0)
gold=train.loc[gold_mask].copy()

print('Studies:',len(train))
print('Fully gold-labeled:',int(gold_mask.sum()))
print('Partially labeled:',int(partial_mask.sum()))
print('No structured labels:',int(unlabeled_mask.sum()))
print('Gold fraction:',float(gold_mask.mean()))
display(train['n_available_labels'].value_counts().sort_index().rename('n_studies').to_frame())

if len(gold):
    prevalence=pd.DataFrame({'positive':gold[LABELS].sum(),'negative':len(gold)-gold[LABELS].sum(),'prevalence':gold[LABELS].mean()}).sort_values('prevalence',ascending=False)
    display(prevalence)
    prevalence['prevalence'].sort_values().plot(kind='barh',figsize=(9,6),title='Gold-label prevalence')
    plt.xlabel('positive fraction'); plt.tight_layout(); plt.show()
    co=gold[LABELS].astype(float).corr(); display(co.round(2))
    fig,ax=plt.subplots(figsize=(9,8)); im=ax.imshow(co.values,aspect='auto');
    ax.set_xticks(range(len(LABELS))); ax.set_xticklabels(LABELS,rotation=90)
    ax.set_yticks(range(len(LABELS))); ax.set_yticklabels(LABELS); ax.set_title('Gold-label correlation'); fig.colorbar(im,ax=ax); plt.tight_layout(); plt.show()
else:
    prevalence=pd.DataFrame(); co=pd.DataFrame()

In [ ]:
# Radiology report audit: duplicates/leakage + rough script, not language detection
def normalize_report(x):
    if pd.isna(x): return ''
    x=str(x).lower().strip(); x=re.sub(r'\s+',' ',x); x=re.sub(r'[^\w\s]','',x,flags=re.UNICODE); return x

def sha1_text(x): return hashlib.sha1(x.encode('utf-8',errors='ignore')).hexdigest()

def dominant_script(text):
    cnt=Counter()
    for ch in str(text):
        if not ch.isalpha(): continue
        try: name=unicodedata.name(ch)
        except ValueError: continue
        if 'LATIN' in name: cnt['Latin']+=1
        elif 'GREEK' in name: cnt['Greek']+=1
        elif 'CYRILLIC' in name: cnt['Cyrillic']+=1
        elif any(k in name for k in ['CJK','HIRAGANA','KATAKANA','HANGUL']): cnt['CJK']+=1
        elif 'ARABIC' in name: cnt['Arabic']+=1
        else: cnt['Other']+=1
    return cnt.most_common(1)[0][0] if cnt else 'Unknown'

report_series=train['Report'].fillna('').astype(str)
train['report_chars']=report_series.str.len(); train['report_words']=report_series.str.split().str.len()
train['report_norm']=report_series.map(normalize_report); train['report_hash']=train['report_norm'].map(sha1_text)
train['report_script']=report_series.map(dominant_script)

dup_counts=train.loc[train['report_norm']!='','report_hash'].value_counts(); duplicate_hashes=dup_counts[dup_counts>1]
dup_rows=train[train['report_hash'].isin(duplicate_hashes.index)].copy(); dup_rows['duplicate_group_size']=dup_rows['report_hash'].map(dup_counts)
dup_rows=dup_rows.sort_values(['duplicate_group_size','report_hash'],ascending=[False,True])

print('Missing/empty reports:',int((train['report_norm']=='').sum()))
print('Unique raw reports:',report_series.nunique(),'Unique normalized:',train['report_norm'].nunique())
print('Duplicate normalized report groups:',len(duplicate_hashes))
print('Studies in duplicate groups:',int(duplicate_hashes.sum()) if len(duplicate_hashes) else 0)
print('Largest duplicate group:',int(duplicate_hashes.max()) if len(duplicate_hashes) else 0)
display(train[['report_chars','report_words']].describe().T)
display(train['report_script'].value_counts().to_frame('n_studies'))
display(dup_rows[['StudyInstanceUID','duplicate_group_size','n_available_labels','report_chars','report_script','Report']].head(30))

In [ ]:
# Series-level EDA
study_series_counts=series.groupby('StudyInstanceUID').size().rename('n_series').reset_index()
display(study_series_counts['n_series'].describe().to_frame().T)
study_series_counts['n_series'].hist(bins=range(int(study_series_counts.n_series.min()),int(study_series_counts.n_series.max())+2),figsize=(8,4)); plt.title('Series per study'); plt.xlabel('n_series'); plt.show()

for col in ['Anatomical_Plane','Fluid_Sensitive','Fat_Suppression']:
    if col in series.columns:
        print('\n',col); display(series[col].value_counts(dropna=False).to_frame('n_series'))

avail=[c for c in ['Anatomical_Plane','Fluid_Sensitive','Fat_Suppression'] if c in series.columns]
if avail: display(series.groupby(avail).size().rename('n_series').reset_index())
if {'Fluid_Sensitive','Fat_Suppression'}.issubset(series.columns):
    print('Fluid_Sensitive != Fat_Suppression:',int((series['Fluid_Sensitive']!=series['Fat_Suppression']).sum()),'/',len(series))
if 'Anatomical_Plane' in series.columns:
    pp=pd.crosstab(series['StudyInstanceUID'],series['Anatomical_Plane']).gt(0)
    for p in ['Sagittal','Coronal','Axial']:
        if p not in pp.columns: pp[p]=False
        print('Missing',p,':',int((~pp[p]).sum()))
    print('All 3 planes:',int(pp[['Sagittal','Coronal','Axial']].all(axis=1).sum()),'/',len(pp))

In [ ]:
# Gold vs non-gold acquisition counts
study_meta=train[['StudyInstanceUID','n_available_labels']].merge(study_series_counts,on='StudyInstanceUID',how='left')
study_meta['is_gold']=study_meta['n_available_labels'].eq(len(LABELS))
if 'Anatomical_Plane' in series.columns:
    plane_n=series.pivot_table(index='StudyInstanceUID',columns='Anatomical_Plane',values='SeriesInstanceUID',aggfunc='count',fill_value=0).add_prefix('n_').reset_index()
    study_meta=study_meta.merge(plane_n,on='StudyInstanceUID',how='left')
numcols=[c for c in study_meta.columns if c.startswith('n_') and c!='n_available_labels']
display(study_meta.groupby('is_gold')[numcols].agg(['count','mean','median','min','max']).round(2))

In [ ]:
# DICOM header scan: one DICOM per sampled series, no pixel decode
import pydicom
print('pydicom:',pydicom.__version__)

def first_dcm_in_dir(d):
    try:
        with os.scandir(d) as it:
            for e in it:
                if e.is_file() and e.name.lower().endswith('.dcm'): return Path(e.path)
    except Exception: return None

def sf(x):
    try: return float(x)
    except Exception: return np.nan

def ss(x):
    try: return None if x is None else str(x)
    except Exception: return None

def mv(ds,name):
    try:
        x=getattr(ds,name,None); return None if x is None else [float(v) for v in x]
    except Exception: return None

samp=series.copy()
if MAX_DICOM_SERIES_HEADERS is not None and len(samp)>MAX_DICOM_SERIES_HEADERS: samp=samp.sample(MAX_DICOM_SERIES_HEADERS,random_state=SEED)
rows=[]; header_errors=[]
for i,r in enumerate(samp.itertuples(index=False),1):
    suid=r.StudyInstanceUID; serid=r.SeriesInstanceUID; f=first_dcm_in_dir(TRAIN_DICOM_ROOT/str(suid)/str(serid))
    if f is None: header_errors.append((suid,serid,'no_dcm')); continue
    try:
        ds=pydicom.dcmread(str(f),stop_before_pixels=True,force=True); ps=mv(ds,'PixelSpacing')
        rows.append({'StudyInstanceUID':suid,'SeriesInstanceUID':serid,'sample_dcm':str(f),
          'Rows':sf(getattr(ds,'Rows',np.nan)),'Columns':sf(getattr(ds,'Columns',np.nan)),
          'PixelSpacingRow':ps[0] if ps and len(ps)>=2 else np.nan,'PixelSpacingCol':ps[1] if ps and len(ps)>=2 else np.nan,
          'SliceThickness':sf(getattr(ds,'SliceThickness',np.nan)),'SpacingBetweenSlices':sf(getattr(ds,'SpacingBetweenSlices',np.nan)),
          'Manufacturer':ss(getattr(ds,'Manufacturer',None)),'ManufacturerModelName':ss(getattr(ds,'ManufacturerModelName',None)),
          'MagneticFieldStrength':sf(getattr(ds,'MagneticFieldStrength',np.nan)),'SeriesDescription':ss(getattr(ds,'SeriesDescription',None)),
          'ProtocolName':ss(getattr(ds,'ProtocolName',None)),'Laterality':ss(getattr(ds,'Laterality',None)),'ImageLaterality':ss(getattr(ds,'ImageLaterality',None)),
          'PhotometricInterpretation':ss(getattr(ds,'PhotometricInterpretation',None)),'BitsStored':sf(getattr(ds,'BitsStored',np.nan)),
          'TransferSyntaxUID':ss(getattr(getattr(ds,'file_meta',None),'TransferSyntaxUID',None))})
    except Exception as e: header_errors.append((suid,serid,repr(e)))
    if i%1000==0: print(i,'/',len(samp))
headers=pd.DataFrame(rows)
print('Header rows:',len(headers),'errors:',len(header_errors)); display(headers.head())

In [ ]:
# Scanner/domain + geometry audit
if len(headers):
    for col in ['Manufacturer','ManufacturerModelName','MagneticFieldStrength','PhotometricInterpretation','TransferSyntaxUID']:
        print('\nTOP',col); display(headers[col].value_counts(dropna=False).head(30).to_frame('n_series'))
    gcols=['Rows','Columns','PixelSpacingRow','PixelSpacingCol','SliceThickness','SpacingBetweenSlices','MagneticFieldStrength']
    display(headers[gcols].describe(percentiles=[.01,.05,.1,.25,.5,.75,.9,.95,.99]).T)
    headers['FOV_mm_row']=headers['Rows']*headers['PixelSpacingRow']; headers['FOV_mm_col']=headers['Columns']*headers['PixelSpacingCol']; headers['pixel_area_mm2']=headers['PixelSpacingRow']*headers['PixelSpacingCol']
    display(headers[['FOV_mm_row','FOV_mm_col','pixel_area_mm2']].describe(percentiles=[.01,.05,.25,.5,.75,.95,.99]).T)
    headers['PixelSpacingRow'].dropna().hist(bins=50,figsize=(8,4)); plt.title('Pixel spacing row (mm/pixel)'); plt.show()
    headers['SliceThickness'].dropna().hist(bins=50,figsize=(8,4)); plt.title('Slice thickness (mm)'); plt.show()
    h2=headers.merge(series[['StudyInstanceUID','SeriesInstanceUID','Anatomical_Plane','Fluid_Sensitive','Fat_Suppression']],on=['StudyInstanceUID','SeriesInstanceUID'],how='left')
    display(h2.groupby('Anatomical_Plane')[['PixelSpacingRow','PixelSpacingCol','SliceThickness','FOV_mm_row','FOV_mm_col']].agg(['count','median','mean']).round(3))
    for col in ['SeriesDescription','ProtocolName']:
        print('\nMost common',col); display(headers[col].fillna('Missing').value_counts().head(80).to_frame('n_series'))

In [ ]:
# Slice count sample
sc_samp=series.copy()
if MAX_SERIES_FILE_COUNTS is not None and len(sc_samp)>MAX_SERIES_FILE_COUNTS: sc_samp=sc_samp.sample(MAX_SERIES_FILE_COUNTS,random_state=SEED)
sc=[]; sc_errors=[]
for r in sc_samp.itertuples(index=False):
    d=TRAIN_DICOM_ROOT/str(r.StudyInstanceUID)/str(r.SeriesInstanceUID)
    try:
        n=sum(1 for e in os.scandir(d) if e.is_file() and e.name.lower().endswith('.dcm'))
        sc.append({'StudyInstanceUID':r.StudyInstanceUID,'SeriesInstanceUID':r.SeriesInstanceUID,'n_slices':n})
    except Exception as e: sc_errors.append((r.StudyInstanceUID,r.SeriesInstanceUID,repr(e)))
slice_counts=pd.DataFrame(sc)
print('counted:',len(slice_counts),'errors:',len(sc_errors)); display(slice_counts['n_slices'].describe(percentiles=[.01,.05,.1,.25,.5,.75,.9,.95,.99]).to_frame().T)
if len(slice_counts):
    slice_counts['n_slices'].clip(upper=slice_counts['n_slices'].quantile(.99)).hist(bins=60,figsize=(8,4)); plt.title('Slices per series (clipped p99)'); plt.show()

In [ ]:
# Anatomical slice-order audit using DICOM geometry
# normal = cross(row_direction, col_direction); position = dot(IPP, normal)
def dcm_files(d):
    try: return sorted([Path(e.path) for e in os.scandir(d) if e.is_file() and e.name.lower().endswith('.dcm')],key=lambda p:p.name)
    except Exception: return []

def geo_pos(ds):
    try:
        iop=np.asarray([float(v) for v in ds.ImageOrientationPatient]); ipp=np.asarray([float(v) for v in ds.ImagePositionPatient]); normal=np.cross(iop[:3],iop[3:]); return float(np.dot(ipp,normal))
    except Exception: return np.nan

audit=series.sample(min(MAX_SERIES_SLICE_ORDER_AUDIT,len(series)),random_state=SEED); out=[]
for r in audit.itertuples(index=False):
    files=dcm_files(TRAIN_DICOM_ROOT/str(r.StudyInstanceUID)/str(r.SeriesInstanceUID)); pos=[]
    for f in files:
        try:
            ds=pydicom.dcmread(str(f),stop_before_pixels=True,force=True,specific_tags=['ImageOrientationPatient','ImagePositionPatient']); pos.append(geo_pos(ds))
        except Exception: pos.append(np.nan)
    vp=np.asarray([x for x in pos if np.isfinite(x)])
    if len(vp)>=3:
        dif=np.diff(vp); mono=bool(np.all(dif>=0) or np.all(dif<=0)); dup=int(len(vp)-len(np.unique(np.round(vp,5))))
    else: mono=np.nan; dup=np.nan
    out.append({'StudyInstanceUID':r.StudyInstanceUID,'SeriesInstanceUID':r.SeriesInstanceUID,'n_files':len(files),'n_position_readable':len(vp),'lexical_filename_order_is_anatomically_monotonic':mono,'duplicate_geometric_positions':dup})
slice_order_audit=pd.DataFrame(out); display(slice_order_audit.head())
valid_order=slice_order_audit['lexical_filename_order_is_anatomically_monotonic'].dropna()
if len(valid_order):
    print('Lexicographic filename order anatomically monotonic fraction:',float(valid_order.mean()))
    print('Non-monotonic rate:',float((~valid_order.astype(bool)).mean()))
print('Series with duplicate geometric positions:',int((slice_order_audit['duplicate_geometric_positions'].fillna(0)>0).sum()))

In [ ]:
# Tiny pixel-decode QC
files=headers['sample_dcm'].dropna().tolist() if len(headers) else []
if len(files)>MAX_PIXEL_DECODE: files=list(np.random.default_rng(SEED).choice(files,MAX_PIXEL_DECODE,replace=False))
dq=[]
for f in files:
    try:
        ds=pydicom.dcmread(str(f),force=True); arr=ds.pixel_array
        dq.append({'file':str(f),'ok':True,'shape':str(arr.shape),'dtype':str(arr.dtype),'min':float(np.nanmin(arr)),'max':float(np.nanmax(arr)),'mean':float(np.nanmean(arr)),'error':None})
    except Exception as e:
        dq.append({'file':str(f),'ok':False,'shape':None,'dtype':None,'min':np.nan,'max':np.nan,'mean':np.nan,'error':repr(e)})
decode_qc=pd.DataFrame(dq)
print('Decode success:',int(decode_qc.ok.sum()) if len(decode_qc) else 0,'/',len(decode_qc)); display(decode_qc.loc[~decode_qc.ok].head(20) if len(decode_qc) else decode_qc)

# CV hygiene recommendation
train['cv_report_group']=train['report_hash']; g=train['cv_report_group'].value_counts(); g2=g[g>1]
print('\nCV grouping: normalized report hash')
print('Unique groups:',train['cv_report_group'].nunique(),'multi-study groups:',len(g2),'studies in them:',int(g2.sum()) if len(g2) else 0)
print('Do NOT split identical normalized reports across folds.')

In [ ]:
# Build summary + export artifacts
summary={
 'dataset_root':str(ROOT),'fast_mode':FAST_MODE,'n_train_studies':int(len(train)),'n_train_series':int(len(series)),
 'n_fully_gold_labeled':int(gold_mask.sum()),'n_partially_labeled':int(partial_mask.sum()),'n_no_structured_labels':int(unlabeled_mask.sum()),'gold_fraction':float(gold_mask.mean()),
 'report_unique_raw':int(report_series.nunique()),'report_unique_normalized':int(train.report_norm.nunique()),'report_duplicate_groups':int(len(duplicate_hashes)),
 'report_studies_in_duplicate_groups':int(duplicate_hashes.sum()) if len(duplicate_hashes) else 0,'report_largest_duplicate_group':int(duplicate_hashes.max()) if len(duplicate_hashes) else 0,
 'series_per_study':{'mean':float(study_series_counts.n_series.mean()),'median':float(study_series_counts.n_series.median()),'min':int(study_series_counts.n_series.min()),'max':int(study_series_counts.n_series.max())},
 'dicom_header_sample_n':int(len(headers)),'dicom_header_errors_n':int(len(header_errors)),'slice_count_sample_n':int(len(slice_counts)),'slice_count_errors_n':int(len(sc_errors)),
 'slice_order_audit_n':int(len(slice_order_audit)),'pixel_decode_sample_n':int(len(decode_qc)),'pixel_decode_success_n':int(decode_qc.ok.sum()) if len(decode_qc) else 0
}
if len(gold): summary['gold_label_prevalence']={c:float(gold[c].mean()) for c in LABELS}
if 'Anatomical_Plane' in series.columns: summary['series_plane_counts']={str(k):int(v) for k,v in series.Anatomical_Plane.value_counts(dropna=False).items()}
if {'Fluid_Sensitive','Fat_Suppression'}.issubset(series.columns): summary['fluid_fat_mismatch_n']=int((series.Fluid_Sensitive!=series.Fat_Suppression).sum())
if len(slice_counts): summary['slice_count_sample']={'mean':float(slice_counts.n_slices.mean()),'median':float(slice_counts.n_slices.median()),'min':int(slice_counts.n_slices.min()),'max':int(slice_counts.n_slices.max()),'p95':float(slice_counts.n_slices.quantile(.95)),'p99':float(slice_counts.n_slices.quantile(.99))}
if len(valid_order): summary['lexical_filename_order_anatomically_monotonic_fraction']=float(valid_order.mean())
if len(headers):
    for c in ['PixelSpacingRow','PixelSpacingCol','SliceThickness','MagneticFieldStrength']:
        v=pd.to_numeric(headers[c],errors='coerce').dropna()
        if len(v): summary[c+'_sample']={'median':float(v.median()),'p05':float(v.quantile(.05)),'p95':float(v.quantile(.95))}

OUT=Path('/kaggle/working/rsna_eda_v2') if Path('/kaggle/working').exists() else Path('/mnt/data/rsna_eda_v2'); OUT.mkdir(parents=True,exist_ok=True)
with open(OUT/'rsna_eda_summary.json','w',encoding='utf-8') as f: json.dump(summary,f,indent=2,ensure_ascii=False)
study_series_counts.to_csv(OUT/'study_series_counts.csv',index=False); study_meta.to_csv(OUT/'study_acquisition_summary.csv',index=False)
if len(dup_rows): dup_rows[['StudyInstanceUID','report_hash','duplicate_group_size','n_available_labels','report_chars','report_words','report_script','Report']].to_csv(OUT/'duplicate_reports.csv',index=False)
if len(headers): headers.to_csv(OUT/'dicom_header_sample.csv',index=False)
if len(slice_counts): slice_counts.to_csv(OUT/'slice_count_sample.csv',index=False)
if len(slice_order_audit): slice_order_audit.to_csv(OUT/'slice_order_audit.csv',index=False)
if len(decode_qc): decode_qc.to_csv(OUT/'pixel_decode_qc.csv',index=False)
if len(prevalence): prevalence.to_csv(OUT/'gold_label_prevalence.csv')

lines=['RSNA Knee EDA v2 — RESULTS TO SEND BACK','='*60,f"Train studies: {summary['n_train_studies']}",f"Train series: {summary['n_train_series']}",f"Fully gold-labeled: {summary['n_fully_gold_labeled']}",f"No structured labels: {summary['n_no_structured_labels']}",f"Gold fraction: {summary['gold_fraction']:.6f}",'',f"Duplicate-report groups: {summary['report_duplicate_groups']}",f"Studies in duplicate groups: {summary['report_studies_in_duplicate_groups']}",f"Largest duplicate group: {summary['report_largest_duplicate_group']}",'',f"Series/study mean={summary['series_per_study']['mean']:.2f}, median={summary['series_per_study']['median']:.2f}, min={summary['series_per_study']['min']}, max={summary['series_per_study']['max']}"]
if 'fluid_fat_mismatch_n' in summary: lines.append(f"Fluid_Sensitive != Fat_Suppression: {summary['fluid_fat_mismatch_n']}")
if 'slice_count_sample' in summary:
    s=summary['slice_count_sample']; lines.append(f"Slice count sample: median={s['median']:.1f}, p95={s['p95']:.1f}, p99={s['p99']:.1f}, max={s['max']}")
if 'lexical_filename_order_anatomically_monotonic_fraction' in summary: lines.append(f"Filename-order anatomically monotonic fraction: {summary['lexical_filename_order_anatomically_monotonic_fraction']:.4f}")
lines += ['',f"DICOM header sample: {summary['dicom_header_sample_n']} (errors={summary['dicom_header_errors_n']})",f"Pixel decode: {summary['pixel_decode_success_n']}/{summary['pixel_decode_sample_n']} successful",'','GOLD LABEL PREVALENCE']
if 'gold_label_prevalence' in summary:
    for k,v in summary['gold_label_prevalence'].items(): lines.append(f'- {k}: {v:.4f}')
lines += ['','NEXT: send this TXT or the ZIP back to ChatGPT.']
text='\n'.join(lines)
with open(OUT/'RESULTS_TO_SEND_BACK.txt','w',encoding='utf-8') as f: f.write(text)
print(text)
ZIP=OUT.parent/'RSNA_Knee_EDA_Artifacts.zip'
with zipfile.ZipFile(ZIP,'w',zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob('*'):
        if p.is_file(): z.write(p,arcname=p.relative_to(OUT))
print('\nArtifacts:',OUT); print('ZIP:',ZIP)